# Multi-Agent RL Notebook

> Hands-on Build It and Exercises.

## Build It

This lesson uses a 6×6 GridWorld with two cooperative agents. They start in opposite corners and must reach a shared goal. Shared reward: `-1` per step while either agent is still moving, `+10` when both arrive. See `code/main.py`.

### Step 1: the multi-agent env

In [ ]:
```python

class CoopGridWorld:

    def __init__(self):

        self.size = 6

        self.goal = (5, 5)

    def reset(self):

        return ((0, 0), (5, 0))  # two agents

    def step(self, state, actions):

        a1, a2 = state

        new1 = move(a1, actions[0])

        new2 = move(a2, actions[1])

        done = (new1 == self.goal) and (new2 == self.goal)

        reward = 10.0 if done else -1.0

        return (new1, new2), reward, done

In [ ]:
```

The *joint* action space is `|A|² = 16`. The global state is two positions.

### Step 2: independent Q-learning

Each agent runs its own Q-table keyed on joint state. At each step: both pick ε-greedy actions, collect joint transition, each updates its own Q with the shared reward.

In [ ]:
```python

def independent_q(env, episodes, alpha, gamma, epsilon):

    Q1, Q2 = defaultdict(default_q), defaultdict(default_q)

    for _ in range(episodes):

        s = env.reset()

        while not done:

            a1 = epsilon_greedy(Q1, s, epsilon)

            a2 = epsilon_greedy(Q2, s, epsilon)

            s_next, r, done = env.step(s, (a1, a2))

            target1 = r + gamma * max(Q1[s_next].values())

            target2 = r + gamma * max(Q2[s_next].values())

            Q1[s][a1] += alpha * (target1 - Q1[s][a1])

            Q2[s][a2] += alpha * (target2 - Q2[s][a2])

            s = s_next

In [ ]:
```

Works on this task because rewards are dense and aligned. Fails on tightly-coupled tasks (e.g., where one agent has to *wait* for the other).

### Step 3: centralized Q with decomposed-value update

Use one Q over joint actions `Q(s, a_1, a_2)`. Update from shared reward. Decentralize at execution by marginalizing: `π_i(s) = argmax_{a_i} max_{a_{-i}} Q(s, a_1, a_2)`. Trades exponential joint action space for a *correct* global view.

### Step 4: simple self-play (adversarial 2-agent)

Same agent, two roles. Train agent A against agent B; after `K` episodes, copy A's weights into B. Symmetric training, consistent progress. The AlphaZero recipe in miniature.

## Exercises

In [ ]:
1. **Easy.** Train independent Q-learning on the 2-agent cooperative GridWorld. How many episodes until mean return > 0? Plot the joint learning curve.
2. **Medium.** Add a "coordination" task: the goal is reached only when both agents step onto it on the same turn. Does independent Q still converge? What breaks?
3. **Hard.** Implement a centralized critic for MAPPO-style training and compare convergence speed to independent PPO on the coordination task.